# ระบบตรวจจับและบันทึกการเทคอนกรีตเสา

Notebook นี้นำโมเดล **Worker** และ **Bucket** ที่ฝึกเสร็จแล้วมาใช้กับวิดีโอสองมุมกล้อง เพื่อตรวจจับกิจกรรมการเท ระบุตำแหน่งเสา บันทึกเวลา และสร้างวิดีโอผลลัพธ์พร้อมผังสถานะเสา 12 ตำแหน่ง

- กล้องด้านข้าง (SIDE): ตรวจจับ Worker และ Bucket เพื่อยืนยันกิจกรรมการเทและระบุ Row
- กล้องด้านหน้า (FRONT): ตรวจจับ Bucket เพื่อระบุ Column
- ตำแหน่งเสา: จับคู่ Row และ Column เช่น `R04_C02`

เตรียมไฟล์โมเดลทั้งสองตัวและวิดีโอไว้ใน Google Drive แล้วรันเซลล์จากบนลงล่าง โดยตรวจสอบพาธในขั้นตอนที่ 3 ให้ตรงกับไฟล์ของตนเอง


# Concrete Pouring Detection — 12-pillar status board

- SIDE ยืนยัน POURING และ Row; FRONT ระบุ Column เท่านั้น; ไม่ใช้ Pouring Zone
- ใช้เงื่อนไขตรวจจับเดิม: Bucket 0.50, Worker 0.45, START 6s/75%, STOP grace 8s, Row switch 5s/75%, event ขั้นต่ำ 10s
- เพิ่มการยืนยัน Column switch 5s/75% เพื่อแยกเสาใน Row เดียวกัน
- แสดงผัง 6 Row × 2 Column ใต้วิดีโอ: ยังไม่เริ่ม / กำลังเท / พักระหว่างรอบ / เทเสร็จ
- เสาเดิมเป็น “เทเสร็จ” เมื่อเสาใหม่ต่างตำแหน่งกัน (`Rxx_Cxx`) มีการเทที่ผ่านเกณฑ์ event ขั้นต่ำแล้วเท่านั้น; เทเสาเดิมซ้ำกี่รอบก็ยังไม่เสร็จ
- สถานะเป็นผลสรุป **ภายในวิดีโอที่ประมวลผล** ไม่ได้พิสูจน์ทางกายภาพว่าเทคอนกรีตเต็มเสา
- หากอ่าน Column ไม่ได้ จะไม่อัปเดตสถานะผังของเสานั้น; event ยังบันทึก `UNKNOWN` ได้ตามระบบเดิม
- CSV หลักมี 8 คอลัมน์: `Event_No, Row, Column, Location, Start_Time, End_Time, Duration_Min, Total_Accumulated_Min`; ข้อมูลแต่ละแถวตรงกับหัวตาราง
- จัดคำอธิบายสี 4 สถานะเป็นช่องกว้างเท่ากัน และจัดกึ่งกลางแนวตั้งของสี่เหลี่ยมสีกับข้อความ; ใช้สีเดียวกับผังสถานะ
- เพิ่ม `pillar_pouring_times.csv` สรุปเวลาเทสะสมต่อเสา (นาที) ทั้ง 12 ตำแหน่ง พร้อมจำนวนรอบและสถานะ โดยคง CSV หลักไว้ 8 คอลัมน์
- เวลาเทสะสม = ผลรวมระยะเวลา event ที่ผ่านเกณฑ์ของเสานั้น ไม่รวมเวลาพักระหว่างรอบ และไม่ต้องกรอกปริมาตรคอนกรีต
- `Duration_Min` ใน CSV หลักคือเวลาเทของรอบนั้น; `Total_Accumulated_Min` คือเวลารวมของเสานั้นถึงรอบที่บันทึก ส่วน CSV สรุปแสดงเวลารวมเมื่อประมวลผลวิดีโอจบ
- สรุปเวลาที่ตรวจวัดได้ของเสาที่เทค้างไว้ด้วย โดยคงสถานะเดิม เช่น `PAUSED`; สถานะ `DONE` ยังคงต้องยืนยันจากการไปเทเสาถัดไปตามเงื่อนไขเดิม เสาที่ยังไม่เริ่มจะมีเวลา 0 นาที
- Event ที่ไม่ทราบ Column ยังอยู่ใน CSV หลักเป็น `UNKNOWN` แต่ไม่รวมเข้ากับเสาใดเสาหนึ่งในผัง 12 ตำแหน่ง
- คงโมเดล พิกัด ROI ค่าความเชื่อมั่น เงื่อนไขเริ่ม/หยุด/เปลี่ยนเสา และรูปแบบวิดีโอเดิม


## 1. ติดตั้งไลบรารีและฟอนต์ภาษาไทย

ติดตั้ง Ultralytics สำหรับใช้โมเดล YOLO, OpenCV สำหรับอ่านและเขียนวิดีโอ และ Pillow สำหรับวาดผังสถานะ พร้อมติดตั้งฟอนต์ภาษาไทยสำหรับข้อความบนวิดีโอ


In [1]:
!pip install -q ultralytics opencv-python pillow
!apt-get -qq update && apt-get -qq install -y fonts-thai-tlwg >/dev/null


W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


## 2. เชื่อมต่อ Google Drive

เชื่อมต่อ Google Drive เพื่ออ่านไฟล์โมเดลและวิดีโอ รวมถึงบันทึกผลลัพธ์ เมื่อระบบขอสิทธิ์ ให้เลือกบัญชีที่เก็บไฟล์ของโปรเจค


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. กำหนดพาธไฟล์ ROI และค่าของระบบ

ตรวจสอบตัวแปรพาธในส่วน `PATHS` ให้ตรงกับไฟล์ที่ต้องการใช้งาน

| ตัวแปร | ไฟล์หรือโฟลเดอร์ที่ใช้ |
| --- | --- |
| WORKER_MODEL_PATH | โมเดล Worker ที่ฝึกเสร็จแล้ว |
| BUCKET_MODEL_PATH | โมเดล Bucket ที่ฝึกเสร็จแล้ว |
| SIDE_VIDEO_PATH | วิดีโอกล้องด้านข้าง |
| FRONT_VIDEO_PATH | วิดีโอกล้องด้านหน้า |
| OUTPUT_DIR | โฟลเดอร์บันทึกวิดีโอและ CSV ผลลัพธ์ |

กำหนด `column_rois` เป็นกรอบหมุนสำหรับ Column ในกล้องด้านหน้า และ `row_rois` เป็นกรอบสี่เหลี่ยมสำหรับ Row ในกล้องด้านข้าง โดยใช้พิกัดที่ได้จากไฟล์กำหนด ROI ของแต่ละมุมกล้อง

ค่าปัจจุบันในไฟล์ใช้ **Worker confidence = 0.45** และ **Bucket confidence = 0.50** และตรวจจับทุก 5 เฟรม (`FRAME_SKIP = 5`) ขนาดภาพตรวจจับ Worker คือ 960 และ Bucket คือ 1280

ส่วน `TEMPORAL` กำหนดเงื่อนไขยืนยันกิจกรรมตามเวลา ได้แก่ เริ่มเทด้วยหน้าต่าง 6 วินาทีและสัดส่วนกิจกรรมอย่างน้อย 75%, หยุดเมื่อไม่พบกิจกรรมครบ 8 วินาที, ยืนยันการเปลี่ยน Row หรือ Column ด้วยหน้าต่าง 5 วินาทีและสัดส่วนอย่างน้อย 75% และใช้ event ที่มีระยะเวลาอย่างน้อย 10 วินาที

ท้ายเซลล์จะแสดง `True` หรือ `False` เพื่อตรวจสอบว่าพาธโมเดลและวิดีโอทั้งสี่รายการมีไฟล์อยู่หรือไม่


In [3]:
import os
import cv2
import csv
import numpy as np
from collections import deque, Counter, defaultdict
from PIL import Image, ImageDraw, ImageFont
from glob import glob
from functools import lru_cache
from ultralytics import YOLO

# ===================== PATHS =====================
WORKER_MODEL_PATH = "/content/drive/MyDrive/MOdeltest/v8s/best.pt"
# แก้ path นี้ให้เป็นโมเดล Bucket ตัวใหม่ของคุณ
BUCKET_MODEL_PATH = "/content/drive/MyDrive/MOdeltest/bucket/best (2).pt"
SIDE_VIDEO_PATH = "/content/drive/MyDrive/Video_PRO/side19.mp4"
FRONT_VIDEO_PATH = "/content/drive/MyDrive/Video_PRO/front19.mp4"
OUTPUT_DIR = "/content/drive/MyDrive/Video_PRO"
os.makedirs(OUTPUT_DIR, exist_ok=True)
CSV_PATH = os.path.join(OUTPUT_DIR, "pouring_logs_row_column2.csv")
PILLAR_TIME_CSV_PATH = os.path.join(OUTPUT_DIR, "pillar_pouring_times.csv")
DEBUG_VIDEO_PATH = os.path.join(OUTPUT_DIR, "debug_row_column_status_12pillars2.mp4")

# ===================== ROI =====================
column_rois = {
    "C01": {'cx': 478, 'cy': 313, 'w': 121, 'h': 510, 'angle': 13},
    "C02": {'cx': 811, 'cy': 379, 'w': 130, 'h': 631, 'angle': -25},
}
row_rois = {
    "R01": [0, 342, 78, 510],
    "R02": [122, 342, 336, 510],
    "R03": [459, 342, 614, 510],
    "R04": [668, 342, 766, 510],
    "R05": [836, 373, 954, 496],
    "R06": [970, 370, 1161, 499],
}

# ===================== YOLO =====================
WORKER_CLASS_ID = 0
BUCKET_CLASS_ID = 0
WORKER_CONF = 0.45
BUCKET_CONF = 0.50
WORKER_IMG_SIZE = 960
BUCKET_IMG_SIZE = 1280
FRAME_SKIP = 5

# ===================== SIDE =====================
BUCKET_ROW_MARGIN = 0
WORKER_ROW_MARGIN = 80

# ===================== TEMPORAL =====================
START_WINDOW_SEC = 6.0
START_ACTIVITY_RATIO = 0.75
STOP_GRACE_SEC = 8.0
ROW_SWITCH_WINDOW_SEC = 5.0
ROW_SWITCH_ACTIVITY_RATIO = 0.75
MIN_EVENT_DURATION_SEC = 10.0
PILLAR_SWITCH_WINDOW_SEC = 5.0
PILLAR_SWITCH_ACTIVITY_RATIO = 0.75

# ===================== FRONT COLUMN =====================
COLUMN_WINDOW_SEC = 2.0
COLUMN_MAJORITY_RATIO = 0.60
FRONT_COLUMN_MARGIN = 25

SAVE_DEBUG_VIDEO = True
PILLAR_MAP = {}

print('Worker model:', os.path.exists(WORKER_MODEL_PATH))
print('Bucket model:', os.path.exists(BUCKET_MODEL_PATH))
print('SIDE video:', os.path.exists(SIDE_VIDEO_PATH))
print('FRONT video:', os.path.exists(FRONT_VIDEO_PATH))


Worker model: True
Bucket model: True
SIDE video: True
FRONT video: True


## 4. โหลดโมเดลและเตรียมเงื่อนไขตรวจจับ

โหลดโมเดล Worker และ Bucket แยกกัน พร้อมแสดงชื่อคลาสของแต่ละโมเดล จากนั้นเตรียมฟังก์ชันตรวจจับ ตรวจสอบตำแหน่งใน ROI ยืนยันกิจกรรมตามเวลา และบันทึกผล

ในกล้องด้านข้าง ใช้จุดกึ่งกลางขอบล่างของกรอบ Bucket และจุดศูนย์กลางกรอบ Worker ตรวจสอบกิจกรรมใน Row เดียวกัน โดยใช้ระยะเผื่อ Worker 80 พิกเซลตามค่าที่กำหนด ส่วนกล้องด้านหน้าใช้ตำแหน่ง Bucket ตรวจสอบกรอบ Column แบบหมุนและยืนยัน Column จากประวัติการตรวจจับ

ระบบเตรียมสถานะ 12 ตำแหน่งจาก 6 Row × 2 Column เมื่อการเทที่เสาใหม่ต่างตำแหน่งผ่านระยะเวลา event ขั้นต่ำแล้ว จึงเปลี่ยนเสาที่เทก่อนหน้าเป็น `DONE` การเทซ้ำที่ตำแหน่งเดิมยังไม่ถือว่าเสานั้นเสร็จ

เซลล์นี้สร้างหัวตาราง CSV และเริ่มตัวแปรสะสมเวลา จำนวนรอบ และสถานะสำหรับการประมวลผลรอบใหม่


In [5]:
model_worker = YOLO(WORKER_MODEL_PATH)
model_bucket = YOLO(BUCKET_MODEL_PATH)
print('Worker classes:', model_worker.names)
print('Bucket classes:', model_bucket.names)

def box_center(box):
    x1,y1,x2,y2 = box
    return ((x1+x2)/2.0, (y1+y2)/2.0)

def bucket_point(box):
    x1,y1,x2,y2 = box
    return ((x1+x2)/2.0, float(y2))

def point_in_rect(point, roi, margin=0):
    px,py = point
    x1,y1,x2,y2 = roi
    return x1-margin <= px <= x2+margin and y1-margin <= py <= y2+margin

def rotated_roi_points(roi, margin=0):
    rect=((float(roi['cx']),float(roi['cy'])),(float(roi['w']+2*margin),float(roi['h']+2*margin)),float(roi['angle']))
    return np.int32(cv2.boxPoints(rect))

def point_in_column(point, roi, margin=0):
    pts=rotated_roi_points(roi, margin).astype(np.float32)
    return cv2.pointPolygonTest(pts,(float(point[0]),float(point[1])),False) >= 0

def detect_workers(frame):
    r=model_worker.predict(frame, conf=WORKER_CONF, imgsz=WORKER_IMG_SIZE, verbose=False)[0]
    out=[]
    for b in r.boxes:
        if int(b.cls[0]) != WORKER_CLASS_ID: continue
        x1,y1,x2,y2=map(int,b.xyxy[0])
        out.append({'box':[x1,y1,x2,y2],'conf':float(b.conf[0])})
    return out

def detect_buckets(frame):
    r=model_bucket.predict(frame, conf=BUCKET_CONF, imgsz=BUCKET_IMG_SIZE, verbose=False)[0]
    out=[]
    for b in r.boxes:
        if int(b.cls[0]) != BUCKET_CLASS_ID: continue
        x1,y1,x2,y2=map(int,b.xyxy[0])
        out.append({'box':[x1,y1,x2,y2],'conf':float(b.conf[0])})
    return out

def get_side_activity(workers,buckets):
    activity={}
    scores={}
    for row_name,roi in row_rois.items():
        bucket_hits=[d for d in buckets if point_in_rect(bucket_point(d['box']),roi,BUCKET_ROW_MARGIN)]
        worker_hits=[d for d in workers if point_in_rect(box_center(d['box']),roi,WORKER_ROW_MARGIN)]
        active=bool(bucket_hits and worker_hits)
        activity[row_name]=active
        if active:
            bc=max(d['conf'] for d in bucket_hits)
            wc=max(d['conf'] for d in worker_hits)
            scores[row_name]=0.70*bc+0.30*wc
        else:
            scores[row_name]=0.0
    active_rows=[r for r,v in activity.items() if v]
    observed=max(active_rows,key=lambda r:scores[r]) if active_rows else None
    return activity,scores,observed

def get_front_column(buckets):
    cands=[]
    for d in buckets:
        p=bucket_point(d['box'])
        for col,roi in column_rois.items():
            if point_in_column(p,roi,FRONT_COLUMN_MARGIN):
                cands.append((col,d['conf']))
    if not cands: return None
    cands.sort(key=lambda x:x[1], reverse=True)
    return cands[0][0]

def prune_history(history,now,keep_sec):
    while history and history[0][0] < now-keep_sec:
        history.popleft()

def values_in_window(history,now,window_sec):
    return [v for t,v in history if t >= now-window_sec]

def bool_ratio(history,now,window_sec):
    vals=values_in_window(history,now,window_sec)
    return (sum(bool(v) for v in vals)/len(vals)) if vals else 0.0

def window_ready(history,now,window_sec,processed_interval):
    rel=[(t,v) for t,v in history if t >= now-window_sec]
    if len(rel)<2: return False
    span=rel[-1][0]-rel[0][0]
    return span >= max(0.0, window_sec-2*processed_interval)

def first_true_time(history,now,window_sec):
    ts=[t for t,v in history if t>=now-window_sec and bool(v)]
    return ts[0] if ts else now

def last_true_time(history,now,window_sec):
    ts=[t for t,v in history if t>=now-window_sec and bool(v)]
    return ts[-1] if ts else now

def stable_column(history,now):
    vals=values_in_window(history,now,COLUMN_WINDOW_SEC)
    if not vals: return None
    valid=[c for c in vals if c is not None]
    if not valid: return None
    col,count=Counter(valid).most_common(1)[0]
    return col if count/len(vals) >= COLUMN_MAJORITY_RATIO else None

def format_time(seconds):
    m=int(seconds//60); s=seconds%60
    return f'{m:02d}:{s:05.2f}'

csv_header=['Event_No','Row','Column','Location','Start_Time','End_Time','Duration_Min','Total_Accumulated_Min']
with open(CSV_PATH,'w',newline='',encoding='utf-8-sig') as f:
    csv.writer(f).writerow(csv_header)

event_no=0
accumulated=defaultdict(float)
round_count=defaultdict(int)
# Key = Rxx_Cxx. These persist until the video ends.
pillar_status={f'{row}_{col}':'NOT_STARTED' for row in row_rois for col in column_rois}
last_valid_pillar=None

def confirm_active_pillar(state):
    """Confirm a real new pillar only after the current event reaches the minimum duration."""
    global last_valid_pillar
    if state['current_confirmed'] or state['active_row'] is None or state['active_column'] is None:
        return
    if state['last_active_time'] is None or state['start_time'] is None:
        return
    if state['last_active_time'] - state['start_time'] < MIN_EVENT_DURATION_SEC:
        return
    location=f"{state['active_row']}_{state['active_column']}"
    if last_valid_pillar is not None and last_valid_pillar != location:
        pillar_status[last_valid_pillar]='DONE'
        print(f'✓ COMPLETED | {last_valid_pillar} -> {location}')
    last_valid_pillar=location
    state['current_confirmed']=True
    pillar_status[location]='POURING'


def dominant_event_column(counts):
    return max(counts,key=counts.get) if counts else None

def save_event(row_name,column_name,start_time,end_time):
    global event_no
    if start_time is None or end_time is None: return False
    duration=end_time-start_time
    if duration < MIN_EVENT_DURATION_SEC:
        print(f'DROP {row_name}/{column_name}: {duration:.2f}s')
        return False
    event_no += 1
    col_text=column_name if column_name is not None else 'UNKNOWN'
    location=f'{row_name}_{col_text}'
    accumulated[location]+=duration
    total=accumulated[location]
    if column_name is not None and location in pillar_status:
        round_count[location]+=1
        pillar_status[location]='PAUSED'
        with open(CSV_PATH,'a',newline='',encoding='utf-8-sig') as f:
                csv.writer(f).writerow([
                    event_no,
                    row_name,
                    col_text,
                    location,
                    format_time(start_time),
                    format_time(end_time),
                    format_time(duration),
                    format_time(total)
                ])
    print(f'✅ EVENT {event_no} | {location} | {duration:.2f}s | สะสม={total/60:.2f} min')
    return True

def export_pillar_pouring_times():
    """Summarize cumulative pouring time for every pillar, in minutes."""
    header=['Row','Column','Location','Status','Round_Count','Total_Accumulated_Min']
    with open(PILLAR_TIME_CSV_PATH,'w',newline='',encoding='utf-8-sig') as f:
        writer=csv.writer(f)
        writer.writerow(header)
        for row in row_rois:
            for col in column_rois:
                location=f'{row}_{col}'
                writer.writerow([row,col,location,pillar_status[location],
                                 round_count[location],format_time(accumulated[location])])
    print('PILLAR POURING TIME CSV:',PILLAR_TIME_CSV_PATH)



Worker classes: {0: 'worker'}
Bucket classes: {0: 'bucket'}


## 5. เตรียมภาพแสดงผลและผังสถานะเสา

เตรียมฟังก์ชันวาดกรอบ Worker, Bucket และ ROI พร้อมข้อความสถานะบนภาพของแต่ละมุมกล้อง จากนั้นปรับความสูงของภาพเพื่อรวมวิดีโอสองมุมไว้ข้างกัน และวาดผังเสา 12 ตำแหน่งด้านล่าง

| สถานะ | ความหมาย |
| --- | --- |
| NOT_STARTED | ยังไม่เริ่ม |
| POURING | กำลังเท |
| PAUSED | พักระหว่างรอบ |
| DONE | เทเสร็จตามเงื่อนไขเปลี่ยนไปเทเสาถัดไปของระบบ |

สถานะดังกล่าวเป็นผลจากกิจกรรมที่ตรวจพบในวิดีโอที่ประมวลผล


In [6]:
def draw_boxes(frame,workers,buckets):
    for d in workers:
        x1,y1,x2,y2=d['box']; cv2.rectangle(frame,(x1,y1),(x2,y2),(0,255,0),2); cv2.putText(frame,f"W {d['conf']:.2f}",(x1,max(20,y1-5)),cv2.FONT_HERSHEY_SIMPLEX,0.45,(0,255,0),1)
    for d in buckets:
        x1,y1,x2,y2=d['box']; cv2.rectangle(frame,(x1,y1),(x2,y2),(255,255,0),2); bx,by=bucket_point(d['box']); cv2.circle(frame,(int(bx),int(by)),6,(255,0,255),-1); cv2.putText(frame,f"B {d['conf']:.2f}",(x1,max(20,y1-5)),cv2.FONT_HERSHEY_SIMPLEX,0.45,(255,255,0),1)
    return frame

def draw_side(frame,workers,buckets,activity,observed_row,active_row,ratios):
    frame=draw_boxes(frame,workers,buckets)
    for row,roi in row_rois.items():
        x1,y1,x2,y2=roi
        if row==active_row: color=(0,255,0); thick=4
        elif row==observed_row: color=(0,255,255); thick=3
        else: color=(0,0,255); thick=2
        cv2.rectangle(frame,(x1,y1),(x2,y2),color,thick)
        cv2.putText(frame,f'{row} A={int(activity[row])} S={ratios[row]:.2f}',(x1,max(20,y1-6)),cv2.FONT_HERSHEY_SIMPLEX,0.42,color,1)
    state_txt='POURING' if active_row is not None else 'IDLE'
    cv2.putText(frame,'SIDE = POURING DECISION',(15,28),cv2.FONT_HERSHEY_SIMPLEX,0.70,(255,255,255),2)
    cv2.putText(frame,f'State: {state_txt}',(15,56),cv2.FONT_HERSHEY_SIMPLEX,0.60,(0,255,255),2)
    cv2.putText(frame,f'Observed Row: {observed_row}',(15,84),cv2.FONT_HERSHEY_SIMPLEX,0.58,(255,255,255),2)
    cv2.putText(frame,f'Active Row: {active_row}',(15,112),cv2.FONT_HERSHEY_SIMPLEX,0.58,(255,255,255),2)
    return frame

def draw_front(frame,buckets,column_now,event_column):
    frame=draw_boxes(frame,[],buckets)
    for col,roi in column_rois.items():
        pts=rotated_roi_points(roi)
        if col==event_column: color=(0,255,0)
        elif col==column_now: color=(0,255,255)
        else: color=(0,0,255)
        cv2.polylines(frame,[pts],True,color,2)
        tx,ty=map(int,pts[np.argmin(pts[:,1])])
        cv2.putText(frame,col,(tx,max(20,ty-5)),cv2.FONT_HERSHEY_SIMPLEX,0.55,color,2)
    cv2.putText(frame,'FRONT = COLUMN ONLY',(15,28),cv2.FONT_HERSHEY_SIMPLEX,0.70,(255,255,255),2)
    cv2.putText(frame,f'Column now: {column_now}',(15,56),cv2.FONT_HERSHEY_SIMPLEX,0.58,(255,255,255),2)
    cv2.putText(frame,f'Event column: {event_column}',(15,84),cv2.FONT_HERSHEY_SIMPLEX,0.58,(0,255,255),2)
    return frame

def resize_height(frame,target_h):
    h,w=frame.shape[:2]
    if h==target_h: return frame
    scale=target_h/h
    return cv2.resize(frame,(int(round(w*scale)),target_h))


# Cache fonts so the video loop does not run fc-match for every frame.
@lru_cache(maxsize=8)
def board_fonts(scale_key):
    scale=scale_key/100.0
    candidates=(glob('/usr/share/fonts/truetype/tlwg/*.ttf')+
                glob('/usr/share/fonts/truetype/noto/NotoSansThai*.ttf')+
                glob('/usr/share/fonts/opentype/noto/NotoSansThai*.otf'))
    if candidates:
        font_path=candidates[0]
        return (ImageFont.truetype(font_path,max(12,int(26*scale))),
                ImageFont.truetype(font_path,max(11,int(21*scale))),
                ImageFont.truetype(font_path,max(14,int(29*scale))),True)
    return (ImageFont.load_default(),)*3+(False,)

# Draw the status board after combining both camera frames. This does not change
# the original coordinates used by YOLO or any ROI.
def status_board(combined, state):
    h,w=combined.shape[:2]
    scale=max(0.65,w/1920.0)
    strip_h=int(round(260*scale))
    canvas=Image.new('RGB',(w,strip_h),(17,24,32))
    draw=ImageDraw.Draw(canvas)
    regular,small,title,thai_font=board_fonts(round(scale*100))

    margin=int(16*scale)
    draw.text((margin,int(16*scale)),('สถานะเสา 12 ตำแหน่ง' if thai_font else '12 PILLAR STATUS'),font=title,fill=(242,246,249))
    active=None
    if state['active_row'] is not None and state['active_column'] is not None:
        active=f"{state['active_row']}_{state['active_column']}"
    if active:
        focus=(f'{active}  กำลังเท  |  รอบ {round_count[active]+1}' if thai_font
               else f'{active}  POURING  |  ROUND {round_count[active]+1}')
        draw.text((int(w*0.55),int(17*scale)),focus,font=regular,fill=(255,221,146))

    gap=int(7*scale)
    group_w=(w-2*margin-5*gap)//6
    row_y=int(64*scale)
    cell_y=[int(99*scale),int(157*scale)]
    cell_h=int(47*scale)
    colors={
        'NOT_STARTED':((40,49,58),(190,199,206)),
        'POURING':((23,106,80),(255,255,255)),
        'PAUSED':((114,83,28),(255,241,205)),
        'DONE':((39,68,90),(225,245,255)),
    }
    for ri,row in enumerate(row_rois):
        x=margin+ri*(group_w+gap)
        draw.text((x+4*scale,row_y),row,font=small,fill=(186,197,205))
        for ci,col in enumerate(column_rois):
            key=f'{row}_{col}'
            current='POURING' if key==active else pillar_status[key]
            bg,fg=colors[current]
            y=cell_y[ci]
            draw.rounded_rectangle((x,y,x+group_w,y+cell_h),radius=max(3,int(5*scale)),fill=bg)
            draw.text((x+int(10*scale),y+int(7*scale)),col,font=regular,fill=fg)
    legend_labels=(
        [('NOT_STARTED','ยังไม่เริ่ม'),('POURING','กำลังเท'),
         ('PAUSED','พักระหว่างรอบ'),('DONE','เทเสร็จ')]
        if thai_font else
        [('NOT_STARTED','NOT STARTED'),('POURING','POURING'),
         ('PAUSED','PAUSED'),('DONE','DONE')]
    )
    # Align swatches and actual text bounds vertically; use four equal groups.
    legend_y=int(220*scale)
    legend_h=strip_h-legend_y-int(8*scale)
    swatch=max(12,int(18*scale))
    lx=margin
    legend_gap=int(24*scale)  # ระยะห่างระหว่างแต่ละสถานะ

    for status,label in legend_labels:
        center_y=legend_y+legend_h/2.0
        swatch_y=int(round(center_y-swatch/2.0))

        draw.rounded_rectangle(
            (lx,swatch_y,lx+swatch,swatch_y+swatch),
            radius=max(2,int(3*scale)),
            fill=colors[status][0]
        )

        text_x=lx+swatch+int(8*scale)
        bbox=draw.textbbox((0,0),label,font=small)
        text_y=int(round(center_y-(bbox[3]-bbox[1])/2.0-bbox[1]))
        draw.text(
            (text_x,text_y),label,
            font=small,fill=(190,199,206)
        )

        lx=text_x+bbox[2]+legend_gap
    strip=cv2.cvtColor(np.asarray(canvas),cv2.COLOR_RGB2BGR)
    return np.vstack((combined,strip))


## 6. ประมวลผลวิดีโอและบันทึกผลลัพธ์

เปิดวิดีโอทั้งสองมุมและอ่านเฟรมคู่กัน ตรวจจับ Worker กับ Bucket ในภาพด้านข้าง และตรวจจับ Bucket ในภาพด้านหน้า เพื่อนำผลมาระบุ Row, Column และติดตามกิจกรรมการเท

ระบบตรวจสอบการเริ่มเท หยุดเท เปลี่ยน Row และเปลี่ยน Column ตามค่าที่ตั้งไว้ พร้อมสะสมเวลาและอัปเดตผังสถานะ เมื่อ `SAVE_DEBUG_VIDEO = True` จะสร้างวิดีโอที่รวมภาพสองมุมพร้อมกรอบตรวจจับและผังสถานะ

เลือกวิดีโอสองมุมที่เริ่มจากเวลาเดียวกันและมีอัตราเฟรมตรงกัน เพราะโค้ดอ่านเฟรมคู่กันตามลำดับ เมื่อวิดีโอตัวใดตัวหนึ่งอ่านต่อไม่ได้ การประมวลผลจะสิ้นสุดลง

เมื่อทำงานเสร็จ จะแสดงข้อความ `FINISHED` พาธผลลัพธ์ และสรุปสถานะ จำนวนรอบ และเวลาเทสะสมของแต่ละเสา


In [7]:
cap_side=cv2.VideoCapture(SIDE_VIDEO_PATH)
cap_front=cv2.VideoCapture(FRONT_VIDEO_PATH)
if not cap_side.isOpened(): raise RuntimeError('เปิด SIDE video ไม่ได้')
if not cap_front.isOpened(): raise RuntimeError('เปิด FRONT video ไม่ได้')

fps_side=cap_side.get(cv2.CAP_PROP_FPS)
fps_front=cap_front.get(cv2.CAP_PROP_FPS)
if fps_side <= 0 or fps_front <= 0: raise RuntimeError('อ่าน FPS ของวิดีโอไม่ได้')
print(f'SIDE FPS={fps_side:.3f} | FRONT FPS={fps_front:.3f}')
if abs(fps_side-fps_front)>0.2: print('⚠️ FPS สองกล้องต่างกัน ควรตรวจ synchronization')
processed_interval=FRAME_SKIP/fps_side
history_keep_sec=max(START_WINDOW_SEC,STOP_GRACE_SEC,ROW_SWITCH_WINDOW_SEC,
                     COLUMN_WINDOW_SEC,PILLAR_SWITCH_WINDOW_SEC)+2.0

row_history={r:deque() for r in row_rois}
column_history=deque()
column_switch_history=deque()
state={'active_row':None,'active_column':None,'start_time':None,
       'last_active_time':None,'column_counts':Counter(),'current_confirmed':False}

def reset_state():
    state.update(active_row=None,active_column=None,start_time=None,
                 last_active_time=None,column_counts=Counter(),current_confirmed=False)
    column_switch_history.clear()

def event_column():
    return state['active_column'] or dominant_event_column(state['column_counts'])

def begin_event(row,col,start_time,last_active_time):
    state.update(active_row=row,active_column=col,start_time=start_time,
                 last_active_time=last_active_time,column_counts=Counter(),
                 current_confirmed=False)
    if col is not None: state['column_counts'][col]+=1
    column_switch_history.clear()


def candidate_column_switch(now, col_stable, side_activity):
    """New FRONT column must persist while SIDE still confirms the active row."""
    old_col=state['active_column']
    candidate=(col_stable if side_activity and old_col is not None
               and col_stable is not None and col_stable != old_col else None)
    column_switch_history.append((now,candidate))
    prune_history(column_switch_history,now,history_keep_sec)
    if candidate is None or not window_ready(column_switch_history,now,
                                               PILLAR_SWITCH_WINDOW_SEC,processed_interval):
        return None
    values=values_in_window(column_switch_history,now,PILLAR_SWITCH_WINDOW_SEC)
    if values.count(candidate)/len(values) < PILLAR_SWITCH_ACTIVITY_RATIO:
        return None
    start_candidates=[t for t,c in column_switch_history
                      if t>=now-PILLAR_SWITCH_WINDOW_SEC and c==candidate]
    return candidate,start_candidates[0]


debug_writer=None
frame_idx=0
current_time=0.0
print('========== START ==========')

while True:
    ok_side,frame_side=cap_side.read(); ok_front,frame_front=cap_front.read()
    if not ok_side or not ok_front: break
    if frame_idx % FRAME_SKIP != 0:
        frame_idx += 1; continue
    current_time=cap_side.get(cv2.CAP_PROP_POS_MSEC)/1000.0

    workers_side=detect_workers(frame_side)
    buckets_side=detect_buckets(frame_side)
    buckets_front=detect_buckets(frame_front)
    activity,row_scores,observed_row=get_side_activity(workers_side,buckets_side)

    for row in row_rois:
        hist=row_history[row]
        hist.append((current_time,bool(activity[row])))
        prune_history(hist,current_time,history_keep_sec)

    column_now=get_front_column(buckets_front)
    column_history.append((current_time,column_now))
    prune_history(column_history,current_time,history_keep_sec)
    col_stable=stable_column(column_history,current_time)
    ratios={row:bool_ratio(hist,current_time,START_WINDOW_SEC)
            for row,hist in row_history.items()}

    if state['active_row'] is None:
        candidates=[]
        for row,hist in row_history.items():
            if window_ready(hist,current_time,START_WINDOW_SEC,processed_interval) and ratios[row] >= START_ACTIVITY_RATIO:
                candidates.append(row)
        if candidates:
            start_row=max(candidates,key=lambda r:(ratios[r],row_scores.get(r,0.0)))
            start_time=first_true_time(row_history[start_row],current_time,START_WINDOW_SEC)
            begin_event(start_row,col_stable,start_time,
                        last_true_time(row_history[start_row],current_time,START_WINDOW_SEC))
            print(f'▶️ START | {start_row}/{col_stable} | @ {start_time:.2f}s | ratio={ratios[start_row]:.2f}')
    else:
        active_row=state['active_row']
        if activity.get(active_row,False): state['last_active_time']=current_time
        if col_stable is not None: state['column_counts'][col_stable]+=1
        if state['active_column'] is None and col_stable is not None:
            state['active_column']=col_stable
            print(f'📍 COLUMN RESOLVED | {active_row}/{col_stable}')

        switch_candidates=[]
        for row,hist in row_history.items():
            if row==active_row: continue
            ratio=bool_ratio(hist,current_time,ROW_SWITCH_WINDOW_SEC)
            if window_ready(hist,current_time,ROW_SWITCH_WINDOW_SEC,processed_interval) and ratio >= ROW_SWITCH_ACTIVITY_RATIO:
                switch_candidates.append((row,ratio))

        if switch_candidates:
            new_row,new_ratio=max(switch_candidates,key=lambda item:(item[1],row_scores.get(item[0],0.0)))
            transition_time=first_true_time(row_history[new_row],current_time,ROW_SWITCH_WINDOW_SEC)
            transition_time=max(state['start_time'],transition_time)
            confirm_active_pillar(state)
            save_event(active_row,event_column(),state['start_time'],transition_time)
            print(f'🔄 ROW SWITCH | {active_row} -> {new_row} @ {transition_time:.2f}s')
            begin_event(new_row,col_stable,transition_time,
                        last_true_time(row_history[new_row],current_time,ROW_SWITCH_WINDOW_SEC))
        else:
            changed=candidate_column_switch(current_time,col_stable,activity.get(active_row,False))
            if changed is not None:
                new_col,transition_time=changed
                transition_time=max(state['start_time'],transition_time)
                confirm_active_pillar(state)
                save_event(active_row,event_column(),state['start_time'],transition_time)
                print(f'🔄 COLUMN SWITCH | {active_row}/{state["active_column"]} -> {active_row}/{new_col} @ {transition_time:.2f}s')
                begin_event(active_row,new_col,transition_time,current_time)
            else:
                last_active=state['last_active_time']
                if last_active is not None and current_time-last_active >= STOP_GRACE_SEC:
                    confirm_active_pillar(state)
                    save_event(active_row,event_column(),state['start_time'],last_active)
                    print(f'⏹️ END | {active_row} | Column={event_column()} @ {last_active:.2f}s')
                    reset_state()

    if state['active_row'] is not None:
        confirm_active_pillar(state)

    if SAVE_DEBUG_VIDEO:
        debug_side=draw_side(frame_side.copy(),workers_side,buckets_side,activity,
                             observed_row,state['active_row'],ratios)
        debug_front=draw_front(frame_front.copy(),buckets_front,col_stable,event_column())
        target_h=max(debug_side.shape[0],debug_front.shape[0])
        debug_side=resize_height(debug_side,target_h)
        debug_front=resize_height(debug_front,target_h)
        combined=np.hstack((debug_side,debug_front))
        combined=status_board(combined,state)
        if debug_writer is None:
            out_h,out_w=combined.shape[:2]
            debug_writer=cv2.VideoWriter(DEBUG_VIDEO_PATH,cv2.VideoWriter_fourcc(*'mp4v'),
                                         fps_side/FRAME_SKIP,(out_w,out_h))
            if not debug_writer.isOpened(): raise RuntimeError('สร้างวิดีโอ debug ไม่ได้')
        debug_writer.write(combined)

    if frame_idx % (FRAME_SKIP*30)==0:
        print(f'{current_time:8.2f}s | observed={observed_row} | active={state["active_row"]} | column_now={column_now} | column_stable={col_stable}')
    frame_idx += 1

if state['active_row'] is not None:
    confirm_active_pillar(state)
    end_time=state['last_active_time'] if state['last_active_time'] is not None else current_time
    save_event(state['active_row'],event_column(),state['start_time'],end_time)

cap_side.release(); cap_front.release()
if debug_writer is not None: debug_writer.release()
export_pillar_pouring_times()
print('========== FINISHED ==========')
print('CSV:',CSV_PATH)
print('DEBUG:',DEBUG_VIDEO_PATH)
print('FINAL STATUS:')
for row in row_rois:
    for col in column_rois:
        loc=f'{row}_{col}'
        print(f'{loc}: {pillar_status[loc]} | rounds={round_count[loc]} | total={accumulated[loc]:.2f}s')


SIDE FPS=30.000 | FRONT FPS=30.000
========== START ==========
    0.00s | observed=None | active=None | column_now=None | column_stable=None
    5.00s | observed=None | active=None | column_now=None | column_stable=None
   10.00s | observed=None | active=None | column_now=None | column_stable=None
   15.00s | observed=None | active=None | column_now=None | column_stable=None
   20.00s | observed=None | active=None | column_now=None | column_stable=None
   25.00s | observed=None | active=None | column_now=None | column_stable=None
   30.00s | observed=None | active=None | column_now=None | column_stable=None
   35.00s | observed=None | active=None | column_now=None | column_stable=None
   40.00s | observed=None | active=None | column_now=None | column_stable=None
   45.00s | observed=None | active=None | column_now=None | column_stable=None
   50.00s | observed=None | active=None | column_now=None | column_stable=None
   55.00s | observed=None | active=None | column_now=None | column_s

## ไฟล์ผลลัพธ์

ไฟล์จะถูกบันทึกในโฟลเดอร์ `OUTPUT_DIR` ซึ่งปัจจุบันคือ `/content/drive/MyDrive/Video_PRO`

| ไฟล์ | เนื้อหา |
| --- | --- |
| debug_row_column_status_12pillars2.mp4 | วิดีโอสองมุมกล้องพร้อมกรอบตรวจจับและผังสถานะเสา |
| pouring_logs_row_column2.csv | บันทึก event ได้แก่ลำดับ Row, Column, ตำแหน่ง เวลาเริ่ม เวลาสิ้นสุด ระยะเวลา และเวลาเทสะสม |
| pillar_pouring_times.csv | สรุปทั้ง 12 ตำแหน่ง พร้อมสถานะ จำนวนรอบ และเวลาเทสะสม |

เปิดวิดีโอผลลัพธ์จาก Google Drive หลังประมวลผลเสร็จ เซลล์ประมวลผลนี้บันทึกวิดีโอลงไฟล์และแสดงข้อความความคืบหน้าใน Notebook

เวลาใน CSV ใช้รูปแบบ `นาที:วินาที` เช่น `02:08.00` หมายถึง 2 นาที 8 วินาที แม้ชื่อคอลัมน์จะลงท้ายด้วย `_Min`
